Aggregates points of a dataset in a similar area into a new aggregated data point

In [ ]:
# Imports
from typing import Callable, List, Tuple
import os
import pandas as pd

# my scripts
from pain2map import AggregationManager, AggregatedPainData, PainData, Coordinate

In [ ]:
# Constants
BASE_PATH = os.path.join('..', 'data', 'actual')
DATA_FILE = os.path.join(BASE_PATH, 'normalized_ctr.csv')
OUT_FILE = os.path.join(BASE_PATH, 'aggr_ctr.csv')

In [ ]:
# Helper Functions
def aggrdp_to_df(aggr_data: List[AggregatedPainData], origin: str) -> pd.DataFrame:
  data = []
  for data_point in aggr_data:
    data.append({
        'lat': data_point.lat,
        'lon': data_point.lng,
        'value': data_point.val,
        'datatype': 'CO2_Emissions',
        'painorigin': origin
    })
  return pd.DataFrame(data)

In [ ]:
dataset = pd.read_csv(DATA_FILE)

aggr_man = AggregationManager(36, 18)
aggr_data = aggr_man.aggregate(dataset, coor_func=AggregatedPainData.mid_point_coordinate)

In [ ]:
#df_aggr = aggrdp_to_df(aggr_data, origin="Aggr_area-center")
#df_aggr.to_csv(OUT_FILE, index=True, index_label='id')

# Visualizing

In [ ]:
# Imports
import matplotlib.pyplot as plt

In [ ]:
depths = [dp.depth for dp in aggr_data]
plt.plot(depths)

In [ ]:
sort_depths = list(depths)
sort_depths.sort()
plt.plot(sort_depths)

# Prepare Comparison

In [ ]:
class AggrConfig:
  @staticmethod
  def coor_func_from_id(center_func_id: int) -> Tuple[Callable[[List[PainData], Coordinate], Coordinate], str]:
    if center_func_id == 0:
      return None, "area"
    elif center_func_id == 1:
      return AggregatedPainData.mid_point_coordinate, "data-mid"
    elif center_func_id == 2:
      return AggregatedPainData.max_point_coordinate, "data-max"
    else:
      raise Exception(f"Illegal center_func_id={center_func_id}")

  def __init__(self, cols: int, rows: int, center_func: 1):
    self.cols = cols
    self.rows = rows
    self.center_func = center_func

  @property
  def origin(self) -> str:
    return f"Aggr_{self.rows}x{self.cols} {AggrConfig.coor_func_from_id(self.center_func)[1]}-centric"
  
  def aggregate(self) -> pd.DataFrame:
    aggr_man = AggregationManager(self.cols, self.rows)
    aggr_data = aggr_man.aggregate(dataset, coor_func=AggrConfig.coor_func_from_id(self.center_func)[0])
    return aggrdp_to_df(aggr_data, origin=self.origin)
  
  def get_file_path(self) -> str:
    return os.path.join(BASE_PATH, f"aggr_ctr_{self.origin}.csv")


In [ ]:
dataset = pd.read_csv(DATA_FILE)

configs: List[AggrConfig] = [
  AggrConfig(36, 18, 0),
  AggrConfig(36, 18, 1),
  AggrConfig(36, 18, 2),
]

for config in configs:
  df_aggr = config.aggregate()
  df_aggr.to_csv(config.get_file_path(), index=True, index_label='id')

In [ ]:
# combine different configs (and regular layer data) into one csv for the database to load
dfs = []
for config in configs:
  dfs.append(pd.read_csv(config.get_file_path(), index_col=False))

# also add regular layer data
if True:
  df_socioeco = pd.read_csv(os.path.join(BASE_PATH, "normalized_ctr_01.csv"), index_col=False)
  df_phys = pd.read_csv(os.path.join(BASE_PATH, "normalized_ctr_05.csv"), index_col=False)
  df_emo = pd.read_csv(os.path.join(BASE_PATH, "normalized_ctr_10.csv"), index_col=False)
  df_env = pd.read_csv(os.path.join(BASE_PATH, "normalized_ctr_50.csv"), index_col=False)
  dfs += [df_socioeco, df_phys, df_emo, df_env]


dataset = pd.concat(
    [df.drop(columns=["id"]) for df in dfs],
    ignore_index=True
)
dataset.to_csv(OUT_FILE, index=True, index_label="id")